Data Transformation

In [1]:
import os
%pwd

'c:\\Users\\YOGA\\OneDrive\\Documents\\AI Projects\\Text-Summarizer\\research'

In [2]:
os.chdir("../")
%pwd

'c:\\Users\\YOGA\\OneDrive\\Documents\\AI Projects\\Text-Summarizer'

Basic Configuration

In [3]:
from dataclasses import dataclass
from pathlib import Path

@dataclass
class DataTransformationConfig:
    rootDir:Path
    dataPath:Path
    tokenizerName:Path


In [4]:
from src.text_summarizer.constants import *
from src.text_summarizer.utils.common import readYaml


In [5]:
from src.text_summarizer.utils.common import createDir


class ConfigManager:
    def __init__(self,
                 config_path=CONFIG_FILE_PATH,
                 params_filepath=PARAMS_FILE_PATH):
        self.config=readYaml(config_path)
        self.params=readYaml(params_filepath)

        createDir([self.config.artifactsRoots])

    def getDataTransformationConfig(self)-> DataTransformationConfig:
        config= self.config.dataTransformation
        
        createDir([config.rootDir])
        dataTransformationConfig=DataTransformationConfig(
            rootDir=config.rootDir,
            dataPath=config.dataPath,
            tokenizerName=config.tokenizerName
        )
        return dataTransformationConfig


In [6]:
import os
from src.text_summarizer.logging import logger
from transformers import AutoTokenizer
from datasets import load_from_disk


Data Transformation Compponents

In [24]:
class DataTransformation:
    def __init__(self,config:DataTransformationConfig):
        self.config=config
        self.tokenizer=AutoTokenizer.from_pretrained(config.tokenizerName)

    def convertExamplesTofeatures(self,exampleBatch):
        inputEncoding=self.tokenizer(exampleBatch['dialogue'],max_length=1024, truncation = True) 
       # with self.tokenizer.asTargetTokenizer():
        targetEncoding=self.tokenizer(exampleBatch['summary'],max_length=128, truncation= True)

        return{
                'input_ids': inputEncoding['input_ids'],
                'attention_mask': inputEncoding['attention_mask'],
                'labels': targetEncoding['input_ids']
        }
    def convert(self):
        datasetSamsum=load_from_disk(self.config.dataPath)
        datasetSamsumPt= datasetSamsum.map(self.convertExamplesTofeatures)
        datasetSamsumPt.save_to_disk(os.path.join(self.config.rootDir,"samsumDataset"))
        
                                           
        


In [27]:
config=ConfigManager()
dataTransformationConfig=config.getDataTransformationConfig()
dataTrandformation=DataTransformation(config=dataTransformationConfig)
dataTrandformation.convert()

[2026-03-05 14:06:46,253: INFO: common: yaml file:config\config.yaml loaded sucessesful]
[2026-03-05 14:06:46,259: INFO: common: yaml file:params.yaml loaded sucessesful]
[2026-03-05 14:06:46,263: INFO: common: created dir at:artifacts]
[2026-03-05 14:06:46,269: INFO: common: created dir at:artifacts/data_transformation]
[2026-03-05 14:06:46,551: INFO: _client: HTTP Request: HEAD https://huggingface.co/google/pegasus-cnn_dailymail/resolve/main/config.json "HTTP/1.1 307 Temporary Redirect"]
[2026-03-05 14:06:46,563: INFO: _client: HTTP Request: HEAD https://huggingface.co/api/resolve-cache/models/google/pegasus-cnn_dailymail/40d588fdab0cc077b80d950b300bf66ad3c75b92/config.json "HTTP/1.1 200 OK"]
[2026-03-05 14:06:46,801: INFO: _client: HTTP Request: HEAD https://huggingface.co/google/pegasus-cnn_dailymail/resolve/main/tokenizer_config.json "HTTP/1.1 307 Temporary Redirect"]
[2026-03-05 14:06:46,814: INFO: _client: HTTP Request: HEAD https://huggingface.co/api/resolve-cache/models/google

Saving the dataset (1/1 shards): 100%|██████████| 818/818 [00:00<00:00, 28249.12 examples/s]
